In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing import image
import numpy as np
import json

In [2]:
train_dir = "../smartvision_dataset/classification/train"
val_dir = "../smartvision_dataset/classification/val"
test_dir = "../smartvision_dataset/classification/test"

In [3]:
img_size = (224, 224)
batch_size = 32

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    zoom_range=0.2,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2]
)

val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical'
)

val_data = val_datagen.flow_from_directory(
    val_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical'
)

test_data = test_datagen.flow_from_directory(
    test_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

Found 1749 images belonging to 25 classes.
Found 375 images belonging to 25 classes.
Found 375 images belonging to 25 classes.


In [4]:
with open("../models/class_indices.json", "w") as f:
    json.dump(train_data.class_indices, f)

In [5]:
num_classes = train_data.num_classes
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224,224,3))
for layer in base_model.layers:
    layer.trainable = False

In [6]:
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001))(x)
x = layers.Dropout(0.5)(x)
output = layers.Dense(num_classes, activation='softmax')(x)

model = models.Model(inputs=base_model.input, outputs=output)

In [7]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 25)             │         6,42

 Total params: 14,854,489 (56.67 MB)

 Trainable params: 138,777 (542.10 KB)

 Non-trainable params: 14,715,712 (56.14 MB)

In [8]:
callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True),
    ReduceLROnPlateau(factor=0.3, patience=3),
    ModelCheckpoint("../models/VGG16_best.h5", save_best_only=True)
]

In [9]:
model.fit(
    train_data,
    validation_data=val_data,
    epochs=25,
    callbacks=callbacks
)

Epoch 1/25
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.0326 - loss: 4.5099

55/55 ━━━━━━━━━━━━━━━━━━━━ 256s 5s/step - accuracy: 0.0435 - loss: 4.3760 - val_accuracy: 0.0960 - val_loss: 3.9289 - learning_rate: 1.0000e-04
Epoch 2/25
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.0765 - loss: 4.0162

55/55 ━━━━━━━━━━━━━━━━━━━━ 258s 5s/step - accuracy: 0.0898 - loss: 3.8853 - val_accuracy: 0.1547 - val_loss: 3.3497 - learning_rate: 1.0000e-04
Epoch 3/25
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.1286 - loss: 3.6094

55/55 ━━━━━━━━━━━━━━━━━━━━ 259s 5s/step - accuracy: 0.1241 - loss: 3.5610 - val_accuracy: 0.2427 - val_loss: 3.0365 - learning_rate: 1.0000e-04
Epoch 4/25
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.1636 - loss: 3.3414

55/55 ━━━━━━━━━━━━━━━━━━━━ 252s 5s/step - accuracy: 0.1841 - loss: 3.3029 - val_accuracy: 0.3493 - val_loss: 2.8386 - learning_rate: 1.0000e-04
Epoch 5/25
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.2128 - loss: 3.1431

...

55/55 ━━━━━━━━━━━━━━━━━━━━ 224s 4s/step - accuracy: 0.5031 - loss: 2.0766 - val_accuracy: 0.5787 - val_loss: 1.9210 - learning_rate: 1.0000e-04
Epoch 22/25
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.4858 - loss: 2.1159

55/55 ━━━━━━━━━━━━━━━━━━━━ 222s 4s/step - accuracy: 0.4923 - loss: 2.0721 - val_accuracy: 0.5920 - val_loss: 1.8990 - learning_rate: 1.0000e-04
Epoch 23/25
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.5151 - loss: 2.0171

55/55 ━━━━━━━━━━━━━━━━━━━━ 231s 4s/step - accuracy: 0.5220 - loss: 2.0259 - val_accuracy: 0.5893 - val_loss: 1.8824 - learning_rate: 1.0000e-04
Epoch 24/25
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.5164 - loss: 1.9877

55/55 ━━━━━━━━━━━━━━━━━━━━ 224s 4s/step - accuracy: 0.5163 - loss: 2.0263 - val_accuracy: 0.5920 - val_loss: 1.8659 - learning_rate: 1.0000e-04
Epoch 25/25
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.5185 - loss: 2.0009

55/55 ━━━━━━━━━━━━━━━━━━━━ 223s 4s/step - accuracy: 0.5220 - loss: 1.9909 - val_accuracy: 0.5920 - val_loss: 1.8490 - learning_rate: 1.0000e-04


In [10]:
for layer in base_model.layers[-10:]:
    layer.trainable = True

for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(
    train_data,
    validation_data=val_data,
    epochs=25,
    callbacks=callbacks
)

Epoch 1/25
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 9s/step - accuracy: 0.5116 - loss: 1.9594

55/55 ━━━━━━━━━━━━━━━━━━━━ 570s 10s/step - accuracy: 0.5300 - loss: 1.9280 - val_accuracy: 0.6000 - val_loss: 1.7932 - learning_rate: 1.0000e-05
Epoch 2/25
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 9s/step - accuracy: 0.5647 - loss: 1.8324

55/55 ━━━━━━━━━━━━━━━━━━━━ 548s 10s/step - accuracy: 0.5655 - loss: 1.8195 - val_accuracy: 0.6427 - val_loss: 1.7293 - learning_rate: 1.0000e-05
Epoch 3/25
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 9s/step - accuracy: 0.5556 - loss: 1.8445

55/55 ━━━━━━━━━━━━━━━━━━━━ 531s 10s/step - accuracy: 0.5729 - loss: 1.7835 - val_accuracy: 0.6240 - val_loss: 1.6813 - learning_rate: 1.0000e-05
Epoch 4/25
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 10s/step - accuracy: 0.6031 - loss: 1.6927

55/55 ━━━━━━━━━━━━━━━━━━━━ 573s 10s/step - accuracy: 0.6135 - loss: 1.6941 - val_accuracy: 0.6640 - val_loss: 1.6234 - learning_rate: 1.0000e-05
Epoch 5/25
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 9s/step - accuracy: 0.6363 - loss: 1.5877

...

55/55 ━━━━━━━━━━━━━━━━━━━━ 512s 9s/step - accuracy: 0.7896 - loss: 1.1003 - val_accuracy: 0.7547 - val_loss: 1.3224 - learning_rate: 1.0000e-05
Epoch 20/25
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.7908 - loss: 1.0912

55/55 ━━━━━━━━━━━━━━━━━━━━ 465s 8s/step - accuracy: 0.7885 - loss: 1.0787 - val_accuracy: 0.7627 - val_loss: 1.3218 - learning_rate: 1.0000e-05
Epoch 21/25
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.8108 - loss: 1.0048

55/55 ━━━━━━━━━━━━━━━━━━━━ 484s 9s/step - accuracy: 0.7925 - loss: 1.0526 - val_accuracy: 0.7707 - val_loss: 1.2985 - learning_rate: 1.0000e-05
Epoch 22/25
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 9s/step - accuracy: 0.8158 - loss: 1.0041

55/55 ━━━━━━━━━━━━━━━━━━━━ 525s 10s/step - accuracy: 0.8096 - loss: 1.0144 - val_accuracy: 0.7680 - val_loss: 1.2880 - learning_rate: 1.0000e-05
Epoch 23/25
55/55 ━━━━━━━━━━━━━━━━━━━━ 562s 10s/step - accuracy: 0.8182 - loss: 1.0099 - val_accuracy: 0.7680 - val_loss: 1.2983 - learning_rate: 1.0000e-05
Epoch 24/25
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 9s/step - accuracy: 0.8274 - loss: 0.9897

55/55 ━━━━━━━━━━━━━━━━━━━━ 534s 10s/step - accuracy: 0.8205 - loss: 0.9997 - val_accuracy: 0.7733 - val_loss: 1.2812 - learning_rate: 1.0000e-05
Epoch 25/25
55/55 ━━━━━━━━━━━━━━━━━━━━ 548s 10s/step - accuracy: 0.8119 - loss: 1.0112 - val_accuracy: 0.7707 - val_loss: 1.2822 - learning_rate: 1.0000e-05


- Train accuracy is 82 percentage

In [11]:
test_loss, test_acc = model.evaluate(test_data)
print("Test Accuracy:", test_acc)

12/12 ━━━━━━━━━━━━━━━━━━━━ 42s 3s/step - accuracy: 0.7973 - loss: 1.1192
Test Accuracy: 0.7973333597183228


- Test Accuracy is 80 percentage

In [ ]:
model.save("../models/VGG16_model.h5")

In [12]:
def prediction():
    model = models.load_model("../models/VGG16_best.h5")

    img_path = "../smartvision_dataset/detection/images/image_000000.jpg"
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = preprocess_input(img_array)
    img_array = np.expand_dims(img_array, axis=0)

    predictions = model.predict(img_array)

    class_indices = train_data.class_indices
    class_names = list(class_indices.keys())

    predicted_class = class_names[np.argmax(predictions)]

    return predicted_class

In [13]:
pred = prediction()
pred

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step


'dog'